In [0]:
select distinct a.attributes_name, a.attributes_npinum
from com_edp_prd.com_intgr.customer_hco as a
left join com_edp_prd.com_intgr.customer_location as b on a.attributes_vid = b.attributes_vid
where a.attributes_name in ('USF Health (University of South Florida)', 
'Orlando Health Arnold Palmer Hospital for Children', 
'UHealth – University of Miami Health System', 
'Emory University / Children''s Healthcare of Atlanta (CHOA)', 
'UAB Medicine – Medical Genetics', 
'MUSC Health (Medical University of South Carolina)', 
'Greenwood Genetic Center', 
'UF Health Shands Hospital', 
'Boston Children''s Hospital', 
'SUNY Upstate Medical University', 
'Yale New Haven Health', 
'University of Rochester Medical Center', 
'Maine Medical Center', 
'University of Michigan Health', 
'DMC Children''s Hospital of Michigan', 
'Riley Children''s Hospital', 
'Michigan State University Health Care', 
'UK HealthCare (University of Kentucky)', 
'UofL Health', 
'Cincinnati Children''s Hospital Medical Center', 
'Nationwide Children''s Hospital', 
'Akron Children''s Hospital', 
'Children''s Hospital of Philadelphia (CHOP)', 
'UPMC (University of Pittsburgh Medical Center)', 
'WVU Medicine (West Virginia University Medicine)', 
'UNC Health (University of North Carolina Health)', 
'Atrium Health', 
'East Tennessee Children''s Hospital', 
'Vanderbilt University Medical Center (VUMC)', 
'Johns Hopkins Medicine', 
'Children''s National Hospital', 
'Inova Health System (Inova Fairfax Medical Campus)', 
'UVA Health (University of Virginia Health)', 
'Hackensack University Medical Center', 
'St. Joseph''s University Medical Center (Paterson)', 
'Saint Peter''s University Hospital', 
'NewYork-Presbyterian / Weill Cornell Medical Center', 
'NYU Langone Health', 
'NYC Health + Hospitals / Metropolitan', 
'Mount Sinai Health System (The Mount Sinai Hospital)', 
'Ann & Robert H. Lurie Children''s Hospital of Chicago', 
'M Health Fairview (University of Minnesota)', 
'UW Health – University of Wisconsin Medical Foundation', 
'University of Iowa Hospitals & Clinics (UI Health Care)', 
'Children''s Hospital & Medical Center (Omaha)', 
'Children''s Wisconsin', 
'UI Health (University of Illinois Hospital & Health Sciences System)', 
'Mayo Clinic Hospital – Rochester', 
'Washington University School of Medicine in St. Louis', 
'MultiCare Mary Bridge Children''s Hospital', 
'Seattle Children''s', 
'UCSF Benioff Children''s Hospital Oakland', 
'Lucile Packard Children''s Hospital Stanford', 
'Oregon Health & Science University (OHSU)', 
'Randall Children''s Hospital at Legacy Emanuel', 
'UCLA Health', 
'Children''s Hospital of Orange County (CHOC)', 
'UC Davis Health', 
'Rady Children''s Hospital – San Diego', 
'Kaiser Permanente – Southern California', 
'Kaiser Permanente – Northern California', 
'Children''s Hospital Los Angeles (CHLA)', 
'Valley Children''s Hospital (Madera)', 
'Phoenix Children''s', 
'Children''s Mercy Kansas City', 
'Intermountain Primary Children''s Hospital', 
'Children''s Hospital Colorado', 
'UNM Health (University of New Mexico Health)', 
'Children''s Medical Center Dallas (Children''s Health; UT Southwestern)', 
'UTHealth Houston', 
'Oklahoma Children''s Hospital at OU Health', 
'Cook Children''s Medical Center', 
'Arkansas Children''s Hospital (affiliated with UAMS)', 
'CHRISTUS Children''s Hospital – San Antonio')
and a.attributes_npinum is not null

In [0]:
%python

!pip install unidecode
!pip install rapidfuzz
import re
import pandas as pd
from unidecode import unidecode
from rapidfuzz import process, fuzz


df_hco = spark.sql("""
SELECT DISTINCT
  a.attributes_name,
  a.attributes_npinum
FROM com_edp_prd.com_intgr.customer_hco AS a
LEFT JOIN com_edp_prd.com_intgr.customer_location AS b
  ON a.attributes_vid = b.attributes_vid
WHERE a.attributes_npinum IS NOT NULL
""").toPandas()


# Defensive cleanup
df_hco = df_hco.dropna(subset=["attributes_name", "attributes_npinum"]).copy()
df_hco["attributes_name"] = df_hco["attributes_name"].astype(str).str.strip()
df_hco["attributes_npinum"] = df_hco["attributes_npinum"].astype(str).str.strip()

sql_in_block = r"""
'USF Health (University of South Florida)', 
'Orlando Health Arnold Palmer Hospital for Children', 
'UHealth – University of Miami Health System', 
'Emory University / Children''s Healthcare of Atlanta (CHOA)', 
'UAB Medicine – Medical Genetics', 
'MUSC Health (Medical University of South Carolina)', 
'Greenwood Genetic Center', 
'SUNY Upstate Medical University', 
'Yale New Haven Health', 
'University of Rochester Medical Center', 
'Maine Medical Center', 
'University of Michigan Health', 
'DMC Children''s Hospital of Michigan', 
'Riley Children''s Hospital', 
'Michigan State University Health Care', 
'UK HealthCare (University of Kentucky)', 
'UofL Health', 
'Children''s Hospital of Philadelphia (CHOP)', 
'UPMC (University of Pittsburgh Medical Center)', 
'WVU Medicine (West Virginia University Medicine)', 
'UNC Health (University of North Carolina Health)', 
'Atrium Health', 
'Vanderbilt University Medical Center (VUMC)', 
'Johns Hopkins Medicine', 
'Inova Health System (Inova Fairfax Medical Campus)', 
'UVA Health (University of Virginia Health)', 
'St. Joseph''s University Medical Center (Paterson)', 
'Saint Peter''s University Hospital', 
'NewYork-Presbyterian / Weill Cornell Medical Center', 
'NYC Health + Hospitals / Metropolitan', 
'Mount Sinai Health System (The Mount Sinai Hospital)', 
'Ann & Robert H. Lurie Children''s Hospital of Chicago', 
'M Health Fairview (University of Minnesota)', 
'UW Health – University of Wisconsin Medical Foundation', 
'University of Iowa Hospitals & Clinics (UI Health Care)', 
'Children''s Hospital & Medical Center (Omaha)', 
'Children''s Wisconsin', 
'UI Health (University of Illinois Hospital & Health Sciences System)', 
'Mayo Clinic Hospital – Rochester', 
'Washington University School of Medicine in St. Louis', 
'MultiCare Mary Bridge Children''s Hospital', 
'Seattle Children''s', 
'UCSF Benioff Children''s Hospital Oakland', 
'Oregon Health & Science University (OHSU)', 
'Randall Children''s Hospital at Legacy Emanuel', 
'Children''s Hospital of Orange County (CHOC)', 
'UC Davis Health', 
'Rady Children''s Hospital – San Diego', 
'Kaiser Permanente – Southern California', 
'Kaiser Permanente – Northern California', 
'Children''s Hospital Los Angeles (CHLA)', 
'Valley Children''s Hospital (Madera)', 
'Phoenix Children''s', 
'Children''s Mercy Kansas City', 
'Intermountain Primary Children''s Hospital', 
'Children''s Hospital Colorado', 
'UNM Health (University of New Mexico Health)', 
'Children''s Medical Center Dallas (Children''s Health; UT Southwestern)', 
'UTHealth Houston', 
'Oklahoma Children''s Hospital at OU Health', 
'Arkansas Children''s Hospital (affiliated with UAMS)', 
'CHRISTUS Children''s Hospital – San Antonio'
"""

canonical_names = [m.group(1).replace("''", "'").strip()
                   for m in re.finditer(r"'((?:[^']|(?:''))*)'", sql_in_block)]

DASH_PATTERN = re.compile(r"[\u2010-\u2015\u2212-]+")  # various dashes + hyphen
NON_ALNUM_SPACE = re.compile(r"[^a-z0-9\s]")

def normalize_name(s: str) -> str:
    if s is None:
        return ""
    s = unidecode(s)                 # remove accents / smart quotes
    s = s.lower().strip()
    s = DASH_PATTERN.sub("-", s)     # normalize all dashes to '-'
    s = s.replace("&", " and ")
    s = s.replace("+", " and ")
    s = s.replace("/", " ")
    s = NON_ALNUM_SPACE.sub(" ", s)  # drop punctuation
    s = re.sub(r"\s+", " ", s)       # squeeze spaces
    return s.strip()


df = df_hco.copy()
df["norm_name"] = df["attributes_name"].map(normalize_name)

# For each normalized name, pick a representative (mode NPI; fallback to first)
def _mode_or_first(series: pd.Series) -> str:
    m = series.mode(dropna=True)
    return (m.iloc[0] if not m.empty else series.iloc[0])

rep = (
    df.groupby("norm_name")
      .agg(
          matched_table_name=("attributes_name", _mode_or_first),
          matched_npi=("attributes_npinum", _mode_or_first),
          cnt=("attributes_name", "size")
      )
      .reset_index()
)

choices = rep["norm_name"].tolist()  # the search space for fuzzy match
choice_index_to_row = rep.set_index("norm_name")  # quick lookup after match


SCORE_CUTOFF = 85  # adjust as needed (80–90 typical)

rows = []
for canon in canonical_names:
    canon_norm = normalize_name(canon)

    # Best match among choices (token_set_ratio is robust to word order)
    matched_norm, score, _ = process.extractOne(
        query=canon_norm,
        choices=choices,
        scorer=fuzz.token_set_ratio,   # could try fuzz.WRatio as an alternative
        score_cutoff=SCORE_CUTOFF
    ) or (None, None, None)

    if matched_norm is None:
        rows.append({
            "canonical_name": canon,
            "match_score": None,
            "matched_table_name": None,
            "matched_npi": None,
            "note": f"No match >= cutoff ({SCORE_CUTOFF})"
        })
    else:
        picked = choice_index_to_row.loc[matched_norm]
        rows.append({
            "canonical_name": canon,
            "match_score": int(score),
            "matched_table_name": picked["matched_table_name"],
            "matched_npi": picked["matched_npi"],
            "note": f"from {picked['cnt']} table rows sharing this normalized form"
        })

best_matches = pd.DataFrame(rows)

ENFORCE_ONE_TO_ONE = False

if ENFORCE_ONE_TO_ONE:
    tmp = (
        best_matches
        .dropna(subset=["matched_table_name", "match_score"])
        .sort_values(["matched_table_name", "match_score"], ascending=[True, False])
    )
    winners = tmp.drop_duplicates(subset=["matched_table_name"], keep="first")
    # Keep winners + all canonicals that had no match
    no_match = best_matches[best_matches["matched_table_name"].isna()]
    best_matches = pd.concat([winners, no_match], ignore_index=True)


# Example: filter to the successful matches only
best_matches_success = best_matches.dropna(subset=["matched_table_name"]).copy()

# If you want to see what did not map
best_matches_unmapped = best_matches[best_matches["matched_table_name"].isna()].copy()

# Print a quick summary
print(f"Canonical HCOs: {len(canonical_names)}")
print(f"Mapped (>= {SCORE_CUTOFF}): {best_matches_success.shape[0]}")
print(f"Unmapped (< {SCORE_CUTOFF}): {best_matches_unmapped.shape[0]}")

# Peek
best_matches_success.head(10)


In [0]:
%python
best_matches_success.display()

In [0]:
%python

!pip install unidecode
!pip install rapidfuzz
import re
import pandas as pd
from unidecode import unidecode
from rapidfuzz import process, fuzz


df_hco = spark.sql("""
SELECT DISTINCT
  a.attributes_name,
  a.attributes_npinum
FROM com_edp_prd.com_intgr.customer_hco AS a
LEFT JOIN com_edp_prd.com_intgr.customer_location AS b
  ON a.attributes_vid = b.attributes_vid
WHERE a.attributes_npinum IS NOT NULL
""").toPandas()

hco_to_lookup = spark.sql("""select * from com_edp_prd.cmpa_insights_internal_schema.hco_names""")

hco_to_lookup has the columns like 

# Defensive cleanup
df_hco = df_hco.dropna(subset=["attributes_name", "attributes_npinum"]).copy()
df_hco["attributes_name"] = df_hco["attributes_name"].astype(str).str.strip()
df_hco["attributes_npinum"] = df_hco["attributes_npinum"].astype(str).str.strip()



canonical_names = [m.group(1).replace("''", "'").strip()
                   for m in re.finditer(r"'((?:[^']|(?:''))*)'", sql_in_block)]

DASH_PATTERN = re.compile(r"[\u2010-\u2015\u2212-]+")  # various dashes + hyphen
NON_ALNUM_SPACE = re.compile(r"[^a-z0-9\s]")

def normalize_name(s: str) -> str:
    if s is None:
        return ""
    s = unidecode(s)                 # remove accents / smart quotes
    s = s.lower().strip()
    s = DASH_PATTERN.sub("-", s)     # normalize all dashes to '-'
    s = s.replace("&", " and ")
    s = s.replace("+", " and ")
    s = s.replace("/", " ")
    s = NON_ALNUM_SPACE.sub(" ", s)  # drop punctuation
    s = re.sub(r"\s+", " ", s)       # squeeze spaces
    return s.strip()


df = df_hco.copy()
df["norm_name"] = df["attributes_name"].map(normalize_name)

# For each normalized name, pick a representative (mode NPI; fallback to first)
def _mode_or_first(series: pd.Series) -> str:
    m = series.mode(dropna=True)
    return (m.iloc[0] if not m.empty else series.iloc[0])

rep = (
    df.groupby("norm_name")
      .agg(
          matched_table_name=("attributes_name", _mode_or_first),
          matched_npi=("attributes_npinum", _mode_or_first),
          cnt=("attributes_name", "size")
      )
      .reset_index()
)

choices = rep["norm_name"].tolist()  # the search space for fuzzy match
choice_index_to_row = rep.set_index("norm_name")  # quick lookup after match


SCORE_CUTOFF = 85  # adjust as needed (80–90 typical)

rows = []
for canon in canonical_names:
    canon_norm = normalize_name(canon)

    # Best match among choices (token_set_ratio is robust to word order)
    matched_norm, score, _ = process.extractOne(
        query=canon_norm,
        choices=choices,
        scorer=fuzz.token_set_ratio,   # could try fuzz.WRatio as an alternative
        score_cutoff=SCORE_CUTOFF
    ) or (None, None, None)

    if matched_norm is None:
        rows.append({
            "canonical_name": canon,
            "match_score": None,
            "matched_table_name": None,
            "matched_npi": None,
            "note": f"No match >= cutoff ({SCORE_CUTOFF})"
        })
    else:
        picked = choice_index_to_row.loc[matched_norm]
        rows.append({
            "canonical_name": canon,
            "match_score": int(score),
            "matched_table_name": picked["matched_table_name"],
            "matched_npi": picked["matched_npi"],
            "note": f"from {picked['cnt']} table rows sharing this normalized form"
        })

best_matches = pd.DataFrame(rows)

ENFORCE_ONE_TO_ONE = False

if ENFORCE_ONE_TO_ONE:
    tmp = (
        best_matches
        .dropna(subset=["matched_table_name", "match_score"])
        .sort_values(["matched_table_name", "match_score"], ascending=[True, False])
    )
    winners = tmp.drop_duplicates(subset=["matched_table_name"], keep="first")
    # Keep winners + all canonicals that had no match
    no_match = best_matches[best_matches["matched_table_name"].isna()]
    best_matches = pd.concat([winners, no_match], ignore_index=True)


# Example: filter to the successful matches only
best_matches_success = best_matches.dropna(subset=["matched_table_name"]).copy()

# If you want to see what did not map
best_matches_unmapped = best_matches[best_matches["matched_table_name"].isna()].copy()

# Print a quick summary
print(f"Canonical HCOs: {len(canonical_names)}")
print(f"Mapped (>= {SCORE_CUTOFF}): {best_matches_success.shape[0]}")
print(f"Unmapped (< {SCORE_CUTOFF}): {best_matches_unmapped.shape[0]}")

# Peek
best_matches_success.head(10)
